In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.model_selection import train_test_split
from PIL import Image
from torch.cuda.amp import GradScaler, autocast

# --- 1. SETTINGS ---
DATA_ROOT = "/kaggle/input/datasets/muhammadbilal6305/200k-real-vs-ai-visuals-by-mbilal/my_real_vs_ai_dataset/my_real_vs_ai_dataset"
BATCH_SIZE = 64  # per GPU
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("GPUs available:", torch.cuda.device_count())

# --- 2. DATASET & TRANSFORMS ---
def is_image_and_in_target_folder(path):
    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    is_img = path.lower().endswith(valid_extensions)
    parts = path.split(os.sep)
    is_in_target = 'ai_images' in parts or 'real' in parts
    return is_img and is_in_target

data_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(
    root=DATA_ROOT,
    transform=data_transforms,
    is_valid_file=is_image_and_in_target_folder
)

train_idx, temp_idx = train_test_split(list(range(len(full_dataset))), test_size=0.2, random_state=42)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_loader = DataLoader(Subset(full_dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(Subset(full_dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(Subset(full_dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# --- 3. MODEL ---
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, 2))

if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs")
    model = nn.DataParallel(model)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scaler = GradScaler()  # For mixed precision

# --- 4. TRAINING LOOP ---
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    # Validation
    model.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with autocast():
                outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data)
            total_val += labels.size(0)

    # Test evaluation
    correct_test, total_test = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with autocast():
                outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct_test += torch.sum(preds == labels.data)
            total_test += labels.size(0)

    print(f"Epoch {epoch+1} | Loss: {train_loss/len(train_loader):.4f} | "
          f"Val Acc: {correct_val.double()/total_val:.4f} | "
          f"Test Acc: {correct_test.double()/total_test:.4f}")

# --- 5. SAVE MODEL ---
torch.save(model.state_dict(), "drishtiai_resnet18_dataparallel.pth")
print("Model saved as drishtiai_resnet18_dataparallel.pth")



Using device: cuda
GPUs available: 2
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 177MB/s]


Using 2 GPUs


/tmp/ipykernel_55/1369425163.py:59: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # For mixed precision
/tmp/ipykernel_55/1369425163.py:68: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/1369425163.py:82: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_55/1369425163.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1 | Loss: 0.2173 | Val Acc: 0.9279 | Test Acc: 0.9268
Epoch 2 | Loss: 0.1225 | Val Acc: 0.9469 | Test Acc: 0.9488
Epoch 3 | Loss: 0.1006 | Val Acc: 0.9471 | Test Acc: 0.9475
Epoch 4 | Loss: 0.0856 | Val Acc: 0.9567 | Test Acc: 0.9547
Epoch 5 | Loss: 0.0732 | Val Acc: 0.9615 | Test Acc: 0.9577
Epoch 6 | Loss: 0.0655 | Val Acc: 0.9587 | Test Acc: 0.9595
Epoch 7 | Loss: 0.0578 | Val Acc: 0.9678 | Test Acc: 0.9660
Epoch 8 | Loss: 0.0508 | Val Acc: 0.9693 | Test Acc: 0.9688
Epoch 9 | Loss: 0.0470 | Val Acc: 0.9586 | Test Acc: 0.9575
Epoch 10 | Loss: 0.0425 | Val Acc: 0.9654 | Test Acc: 0.9660
Model saved as drishtiai_resnet18_dataparallel.pth


In [6]:
def predict_image(image_path):
    img = Image.open(image_path).convert('RGB')
    preprocess = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    img_t = preprocess(img).unsqueeze(0).to(DEVICE)

    # Load model
    state_dict = torch.load("drishtiai_resnet18_dataparallel.pth", map_location=DEVICE)
    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('module.', '')  # remove 'module.' if present
        new_state_dict[name] = v

    model_eval = models.resnet18(weights=None)
    model_eval.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model_eval.fc.in_features, 2))
    model_eval.load_state_dict(new_state_dict)
    model_eval = model_eval.to(DEVICE)
    model_eval.eval()

    with torch.no_grad():
        output = model_eval(img_t)
        prob = torch.nn.functional.softmax(output, dim=1)
        conf, pred = torch.max(prob, 1)

    status = "AI-Generated" if pred.item() == 0 else "Real"
    print(f"Prediction: {status} ({conf.item()*100:.2f}%)")

In [7]:
predict_image("/kaggle/input/datasets/sakshyamkarki/testing/real.jpeg")
predict_image("/kaggle/input/datasets/sakshyamkarki/testing/test.jpeg")
predict_image("/kaggle/input/datasets/sakshyamkarki/testing/me.jpg")
predict_image("/kaggle/input/datasets/sakshyamkarki/testing/me1.jpg")
predict_image("/kaggle/input/datasets/sakshyamkarki/testing/fake.webp")

predict_image("/kaggle/input/datasets/sakshyamkarki/deepfake/f.jpg")
predict_image("/kaggle/input/datasets/sakshyamkarki/deepfake/r.jpg")


Prediction: Real (98.82%)
Prediction: Real (99.99%)
Prediction: AI-Generated (100.00%)
Prediction: AI-Generated (99.96%)
Prediction: AI-Generated (100.00%)
Prediction: Real (98.07%)
Prediction: Real (100.00%)


In [8]:
predict_image("/kaggle/input/datasets/sakshyamkarki/ai-gen/ai.webp")
predict_image("/kaggle/input/datasets/sakshyamkarki/ai-gen/ai1.webp")
predict_image("/kaggle/input/datasets/sakshyamkarki/ai-gen/ai2.webp")

Prediction: AI-Generated (99.92%)
Prediction: AI-Generated (51.89%)
Prediction: AI-Generated (100.00%)


In [9]:
from IPython.display import FileLink

# Create a clickable link
FileLink("drishtiai_resnet18_dataparallel.pth")

/kaggle/working/drishtiai_resnet18_dataparallel.pth